# Entrenamiento FINAL de la LSTM — TFG Crypto (retorno de ETH)

Notebook para entrenar el modelo **definitivo**, paso a paso, y guardarlo para producción.
Se ejecuta una vez (o cada varios meses para reentrenar). Reproduce la ingeniería de
features del análisis, con el **conjunto de columnas ya fijado** (cruce de los dos
análisis: magnitud + dirección) y la arquitectura elegida.

Orden: cargar datos → features → régimen → split → escalado → secuencias → entrenar →
evaluar en test (una vez) → guardar `lstm_final.pt` + `scaler_lstm.pkl`.

## 1. Setup y configuración

In [35]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler, StandardScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Rutas (ajusta si tu estructura difiere) ---
RUTA_MERGED = "../data/csv/df_merged.csv"
RUTA_REGIM  = "../data/csv/regimenes.csv"
DIR_MODELOS = Path("../models")

# --- Configuración del modelado ---
TARGET_COL = "eth_close_ret"
HORIZON    = 3
SEQ_LEN    = 30
SEED       = 42

np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("Config lista.")

Dispositivo: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Config lista.


## 2. Conjunto de features DEFINITIVO

Cruce de los dos análisis: las que mejor predicen **magnitud** (top val_loss) + las que
mejor predicen **dirección** (top DirAcc) + dinámica reciente. El régimen se añade aparte
(one-hot, por fecha).

In [36]:
FEATURE_COLS = [
    # --- magnitud / niveles (top val_loss) ---
    "eth_cum_ret_30d", "btc_dominance_chg14d", "inflation_chg30", "eth_bb_width",
    "eth_mfi14", "eth_dist_sma200", "alt_dominance_diff", "eth_rsi14",
    "eth_vol_14d", "eth_mcap_ret", "eth_stoch_d",
    # --- direccion / sentimiento (top DirAcc) ---
    "n_miedo_ext_30d", "n_codicia_15d", "presion_ext_neta_15d", "fear_greed_scaled",
    # --- dinamica reciente ---
    "eth_close_ret",
]
print(f"{len(FEATURE_COLS)} features (+ regimen one-hot que se anade despues)")

16 features (+ regimen one-hot que se anade despues)


## 3. Indicadores técnicos (idénticos al análisis)

In [37]:
def rsi(close, n=14):
    delta = close.diff()
    g = delta.where(delta > 0, 0).rolling(n).mean()
    p = (-delta.where(delta < 0, 0)).rolling(n).mean()
    return 100 - (100 / (1 + g / (p + 1e-8)))

def estocastico(df, pfx, n=14, d=3):
    ll = df[f"{pfx}_low"].rolling(n).min()
    hh = df[f"{pfx}_high"].rolling(n).max()
    k = 100 * (df[f"{pfx}_close"] - ll) / (hh - ll + 1e-8)
    return k, k.rolling(d).mean()

def mfi(df, pfx, n=14):
    tp = (df[f"{pfx}_high"] + df[f"{pfx}_low"] + df[f"{pfx}_close"]) / 3
    flujo = tp * df[f"{pfx}_volume"]
    delta = tp.diff()
    fp = flujo.where(delta > 0, 0).rolling(n).sum()
    fn = flujo.where(delta < 0, 0).rolling(n).sum()
    return 100 - (100 / (1 + fp / (fn + 1e-8)))

def bollinger(close, n=20, k=2):
    ma = close.rolling(n).mean(); sd = close.rolling(n).std()
    sup, inf = ma + k*sd, ma - k*sd
    return (close - inf) / (sup - inf + 1e-8), (sup - inf) / (ma + 1e-8)

print("Indicadores definidos.")

Indicadores definidos.


## 4. Carga de datos e ingeniería de features

In [38]:
df_raw = pd.read_csv(RUTA_MERGED, parse_dates=["date"], index_col="date").sort_index()
print(f"df_merged: {df_raw.shape}  ({df_raw.index.min().date()} -> {df_raw.index.max().date()})")

# OHE del sentimiento
ohe = pd.get_dummies(df_raw["FearGreed_Label"], prefix="fng", dtype=int)
ohe.columns = [c.replace(" ", "_") for c in ohe.columns]
df_raw = df_raw.drop(columns=["FearGreed_Label"]).join(ohe)
OHE_COLS = [c for c in df_raw.columns if c.startswith("fng_")]

df = df_raw.copy()
# Bloque A: retornos
for col in ["btc_close","btc_volume","eth_close","eth_volume","btc_mcap","eth_mcap"]:
    df[f"{col}_ret"] = df[col].pct_change() * 100
# Bloque B: dominancias
for col in ["btc_dominance","eth_dominance","alt_dominance"]:
    df[f"{col}_diff"] = df[col].diff()
for col in ["btc_dominance","eth_dominance"]:
    for w in [14,30,60]:
        df[f"{col}_chg{w}d"] = df[col] - df[col].shift(w)
# Bloque C: macro + volatilidad
for col in ["inflation","fed_rate"]:
    df[f"{col}_chg30"] = df[col].diff(30)
for w in [7,14,30]:
    df[f"eth_vol_{w}d"] = df["eth_close_ret"].rolling(w).std()
# Bloque D: osciladores
df["eth_rsi14"] = rsi(df["eth_close"]); df["btc_rsi14"] = rsi(df["btc_close"])
df["eth_stoch_k"], df["eth_stoch_d"] = estocastico(df, "eth")
df["btc_stoch_k"], df["btc_stoch_d"] = estocastico(df, "btc")
df["eth_mfi14"] = mfi(df, "eth"); df["btc_mfi14"] = mfi(df, "btc")
df["eth_bb_pctb"], df["eth_bb_width"] = bollinger(df["eth_close"])
for w in [10,15]:
    df[f"eth_rsi_sobrecompra_{w}d"] = (df["eth_rsi14"] > 70).rolling(w).sum()
    df[f"eth_rsi_sobreventa_{w}d"]  = (df["eth_rsi14"] < 30).rolling(w).sum()
# Bloque E: sentimiento
df["fear_greed_scaled"] = df["fear_greed"] / 100.0
for w in [15,30]:
    fc = [c for c in OHE_COLS if "Fear" in c]; gc = [c for c in OHE_COLS if "Greed" in c]
    fs = sum(df[c].rolling(w).sum() for c in fc); gs = sum(df[c].rolling(w).sum() for c in gc)
    df[f"fear_greed_ratio_{w}d"] = fs / (gs + 1e-8)
for w in [7,15,30]:
    df[f"eth_cum_ret_{w}d"] = df["eth_close_ret"].rolling(w).sum()
# Bloque F: nivel
df["eth_btc_ratio"] = df["eth_close"] / df["btc_close"]
df["eth_drawdown"] = (df["eth_close"] / df["eth_close"].cummax() - 1) * 100
df["btc_drawdown"] = (df["btc_close"] / df["btc_close"].cummax() - 1) * 100
for w in [50,200]:
    df[f"eth_dist_sma{w}"] = (df["eth_close"] / df["eth_close"].rolling(w).mean() - 1) * 100
# Bloque E2: sentimiento acumulado
estados = {"miedo_ext":"fng_Extreme_Fear","miedo":"fng_Fear","neutral":"fng_Neutral",
           "codicia":"fng_Greed","codicia_ext":"fng_Extreme_Greed"}
presentes = {k:v for k,v in estados.items() if v in df.columns}
for w in [15,30,60,90]:
    for nombre,col in presentes.items():
        df[f"n_{nombre}_{w}d"] = df[col].rolling(w).sum()
    if "fng_Extreme_Greed" in df.columns and "fng_Extreme_Fear" in df.columns:
        df[f"presion_ext_neta_{w}d"] = (df["fng_Extreme_Greed"].rolling(w).sum()
                                        - df["fng_Extreme_Fear"].rolling(w).sum())
    df[f"fg_presion_{w}d"] = (df["fear_greed"] - 50).rolling(w).sum()

df_model = df.dropna().copy()
print(f"Tras features y dropna: {df_model.shape}")
assert df_model.isna().sum().sum() == 0

df_merged: (3044, 19)  (2018-02-01 -> 2026-06-02)
Tras features y dropna: (2845, 96)


## 5. Cruce del régimen (one-hot por fecha)

In [39]:
df_model = df_model[[c for c in df_model.columns if not c.startswith("regime_")]].copy()
reg = pd.read_csv(RUTA_REGIM, parse_dates=["date"], index_col="date")
col_reg = "estado_hmm_causal" if "estado_hmm_causal" in reg.columns else "estado_hmm"
ohe_reg = pd.get_dummies(reg[col_reg], prefix="regime", dtype=int)
REGIME_COLS = list(ohe_reg.columns)
df_model = df_model.join(ohe_reg, how="inner")
print(f"Regimen cruzado ({col_reg}): {REGIME_COLS}")
print(f"df_model: {len(df_model)} filas")

feats = [c for c in FEATURE_COLS if c in df_model.columns]
faltan = [c for c in FEATURE_COLS if c not in df_model.columns]
if faltan: print(f"AVISO features no encontradas: {faltan}")
feats_all = feats + REGIME_COLS
print(f"Features: {len(feats)} continuas + {len(REGIME_COLS)} regimen = {len(feats_all)}")

Regimen cruzado (estado_hmm_causal): ['regime_0', 'regime_1', 'regime_2']
df_model: 2845 filas
Features: 16 continuas + 3 regimen = 19


## 6. Split temporal 70/15/15 + escalado selectivo

In [40]:
n = len(df_model); i_tr, i_va = int(n*0.70), int(n*0.85)
df_tr, df_va, df_te = df_model.iloc[:i_tr], df_model.iloc[i_tr:i_va], df_model.iloc[i_va:]
print(f"Train {len(df_tr)} ({df_tr.index.min().date()}->{df_tr.index.max().date()})")
print(f"Val   {len(df_va)} ({df_va.index.min().date()}->{df_va.index.max().date()})")
print(f"Test  {len(df_te)} ({df_te.index.min().date()}->{df_te.index.max().date()})")

# Escalado: continuas con RobustScaler, regimen one-hot SIN escalar
sx = RobustScaler().fit(df_tr[feats].values)
sy = StandardScaler().fit(df_tr[[TARGET_COL]].values)

def build_X(d):
    Xc = sx.transform(d[feats].values)
    Xr = d[REGIME_COLS].values.astype(np.float32)
    return np.hstack([Xc, Xr]).astype(np.float32)
def build_y(d):
    return sy.transform(d[[TARGET_COL]].values).ravel()

X_tr, X_va, X_te = build_X(df_tr), build_X(df_va), build_X(df_te)
y_tr, y_va, y_te = build_y(df_tr), build_y(df_va), build_y(df_te)
print(f"X escalado: train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}")

Train 1991 (2018-08-19->2024-01-30)
Val   427 (2024-01-31->2025-04-01)
Test  427 (2025-04-02->2026-06-02)
X escalado: train (1991, 19)  val (427, 19)  test (427, 19)


## 7. Secuencias deslizantes (SEQ_LEN=30, HORIZON=3)

In [41]:
def crear_secuencias(X, y, seq_len, horizon):
    Xs, ys = [], []
    for i in range(seq_len, len(X) - horizon + 1):
        Xs.append(X[i-seq_len:i]); ys.append(y[i:i+horizon])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

Xs_tr, ys_tr = crear_secuencias(X_tr, y_tr, SEQ_LEN, HORIZON)
Xs_va, ys_va = crear_secuencias(X_va, y_va, SEQ_LEN, HORIZON)
Xs_te, ys_te = crear_secuencias(X_te, y_te, SEQ_LEN, HORIZON)
print(f"Secuencias -> train {Xs_tr.shape}  val {Xs_va.shape}  test {Xs_te.shape}")

def loader(Xs, ys, bs=32, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(Xs), torch.from_numpy(ys)),
                      batch_size=bs, shuffle=shuffle)
dl_tr = loader(Xs_tr, ys_tr, shuffle=True)
dl_va = loader(Xs_va, ys_va)

Secuencias -> train (1959, 30, 19)  val (395, 30, 19)  test (395, 30, 19)


## 8. Arquitectura (LSTM apilada 48→24, dropout 0.35)

Red modesta, pensada para pocas features y poco dato. Si ves overfitting (train baja
mucho y val no), reduce a `hidden_sizes=(32,16)` o sube el dropout.

In [42]:
class LSTMRegressor(nn.Module):
    def __init__(self, n_features, hidden_sizes=(32,), horizon=HORIZON, dropout=0.35):
        super().__init__()
        capas, in_size = [], n_features
        for h in hidden_sizes:
            capas.append(nn.LSTM(in_size, h, batch_first=True)); in_size = h
        self.lstms = nn.ModuleList(capas)
        self.drops = nn.ModuleList([nn.Dropout(dropout) for _ in hidden_sizes])
        self.head = nn.Sequential(
            nn.Linear(hidden_sizes[-1], max(hidden_sizes[-1]//2,1)), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(max(hidden_sizes[-1]//2,1), horizon))
    def forward(self, x):
        out = x
        for lstm, drop in zip(self.lstms, self.drops):
            out, _ = lstm(out); out = drop(out)
        return self.head(out[:, -1, :])

modelo = LSTMRegressor(Xs_tr.shape[2]).to(DEVICE)
n_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(modelo)
print(f"\n{n_params:,} params | ratio params/seq = {n_params/len(Xs_tr):.1f}")

LSTMRegressor(
  (lstms): ModuleList(
    (0): LSTM(19, 32, batch_first=True)
  )
  (drops): ModuleList(
    (0): Dropout(p=0.35, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.35, inplace=False)
    (3): Linear(in_features=16, out_features=3, bias=True)
  )
)

7,363 params | ratio params/seq = 3.8


## 9. Entrenamiento (Huber + Adam + ReduceLROnPlateau + early stopping)

In [43]:
crit = nn.HuberLoss(delta=1.0)
opt = torch.optim.Adam(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)

mejor_val, mejor_estado, sin_mejora = float("inf"), None, 0
hist = {"train": [], "val": []}
for ep in range(120):
    modelo.train(); tl = 0.0
    for xb, yb in dl_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(modelo(xb), yb); loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0); opt.step()
        tl += loss.item() * len(xb)
    tl /= len(dl_tr.dataset)
    modelo.eval(); vl = 0.0
    with torch.no_grad():
        for xb, yb in dl_va:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl += crit(modelo(xb), yb).item() * len(xb)
    vl /= len(dl_va.dataset); sched.step(vl)
    hist["train"].append(tl); hist["val"].append(vl)
    if vl < mejor_val - 1e-5:
        mejor_val = vl; sin_mejora = 0
        mejor_estado = {k: v.detach().cpu().clone() for k,v in modelo.state_dict().items()}
    else:
        sin_mejora += 1
        if sin_mejora >= 55:
            print(f"Early stop en epoca {ep}"); break
    if ep % 10 == 0:
        print(f"  ep {ep:3d}  train={tl:.4f}  val={vl:.4f}")

if mejor_estado: modelo.load_state_dict(mejor_estado)
print(f"\nMejor val_loss: {mejor_val:.6f}")

  ep   0  train=0.3602  val=0.2647
  ep  10  train=0.3550  val=0.2673
  ep  20  train=0.3539  val=0.2684
  ep  30  train=0.3534  val=0.2683
  ep  40  train=0.3534  val=0.2686
  ep  50  train=0.3528  val=0.2687
Early stop en epoca 55

Mejor val_loss: 0.264697


## 10. Evaluación en TEST (una sola vez) + baseline

In [44]:
def metricas(y_true, y_pred):
    yt, yp = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    mae = np.mean(np.abs(yt - yp)); rmse = np.sqrt(np.mean((yt - yp)**2))
    mask = yp != 0
    da = np.mean(np.sign(yt[mask]) == np.sign(yp[mask])) if mask.sum() > 0 else np.nan
    return mae, rmse, da

modelo.eval(); preds, trues = [], []
with torch.no_grad():
    for xb, yb in loader(Xs_te, ys_te):
        preds.append(modelo(xb.to(DEVICE)).cpu().numpy()); trues.append(yb.numpy())
preds, trues = np.concatenate(preds), np.concatenate(trues)
preds_r = sy.inverse_transform(preds.ravel().reshape(-1,1)).reshape(preds.shape)
trues_r = sy.inverse_transform(trues.ravel().reshape(-1,1)).reshape(trues.shape)

mae, rmse, da = metricas(trues_r, preds_r)
mae0, rmse0, _ = metricas(trues_r, np.zeros_like(trues_r))

print("="*55)
print("RESULTADOS EN TEST (modelo final, una sola evaluacion)")
print("="*55)
print(f"  MAE   : {mae:.4f} %   (baseline cero: {mae0:.4f})")
print(f"  RMSE  : {rmse:.4f} %   (baseline cero: {rmse0:.4f})")
print(f"  DirAcc: {da:.4f}      (azar = 0.50)")
print()
print("La magnitud (MAE/RMSE) es lo aprovechable; la direccion ~0.50-0.52")
print("confirma que el signo a 3 dias no es predecible (mercado eficiente).")

RESULTADOS EN TEST (modelo final, una sola evaluacion)
  MAE   : 2.5241 %   (baseline cero: 2.5213)
  RMSE  : 3.6674 %   (baseline cero: 3.6731)
  DirAcc: 0.5131      (azar = 0.50)

La magnitud (MAE/RMSE) es lo aprovechable; la direccion ~0.50-0.52
confirma que el signo a 3 dias no es predecible (mercado eficiente).


## 11. Guardar el modelo + scaler para producción

In [45]:
DIR_MODELOS.mkdir(exist_ok=True)

torch.save({
    "state_dict": modelo.state_dict(),
    "arquitectura": {"n_features": Xs_tr.shape[2], "hidden_sizes": (48,24),
                     "horizon": HORIZON, "dropout": 0.35},
    "feature_cols": feats, "regime_cols": REGIME_COLS,
    "seq_len": SEQ_LEN, "horizon": HORIZON, "target_col": TARGET_COL,
}, DIR_MODELOS / "lstm_final.pt")

joblib.dump({"sx": sx, "sy": sy, "feature_cols": feats, "regime_cols": REGIME_COLS},
            DIR_MODELOS / "scaler_lstm.pkl")

print(f"Guardado: {DIR_MODELOS/'lstm_final.pt'}")
print(f"Guardado: {DIR_MODELOS/'scaler_lstm.pkl'}")
print("\nEstos dos archivos son los que cargara el bot para predecir.")

Guardado: ..\models\lstm_final.pt
Guardado: ..\models\scaler_lstm.pkl

Estos dos archivos son los que cargara el bot para predecir.
